# Engenharia De Modelos

## Objetivos da engenharia de modelos para previsão de churns

A engenharia de modelos para previsão de churn tem como objetivo testar e otimizar o modelo através de testes de diferentes técnicas. Serão aplicados:

- Técnicas de Feature Engineering
- Pipelines de processamento para automação
- Tratamento de dados desbalanceados
- Otimizar hiperparâmetros
- Validação de modelos com técnicas apropriadas

# 1 - Configuração de ambiente e set up de bilbiotecas


In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.experimental import enable_halving_search_cv  # noqa: F401
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score, RepeatedStratifiedKFold, HalvingRandomSearchCV
from sklearn.preprocessing import StandardScaler, RobustScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, accuracy_score, precision_score, recall_score, f1_score
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from scipy.stats import randint, uniform
import mlflow
import mlflow.sklearn
import joblib
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')

## 2. Carregamento Dataset Pré-processado no  [Estágio 1](./01-EDA.ipynb)

In [9]:
# Carregar dataset pré-processado da Aula 2
path = "../data/pre-processed/Telco_customer_churn_preprocessed.csv"

df = pd.read_csv(path)

# Garantir coluna alvo presente e binária (já tratada na Aula 2)
assert 'target' in df.columns, "Coluna 'target' não encontrada no dataset pré-processado."

# Remover colunas de metadados se existirem
for col in ['id', 'dataset']:
    if col in df.columns:
        df.drop(columns=[col], inplace=True)

print(f"Dataset shape (pré-processado): {df.shape}")
print("\nTarget (0=Sem churn, 1=Com churn):")
print(df['target'].value_counts())

df.head()

Dataset shape (pré-processado): (7043, 58)

Target (0=Sem churn, 1=Com churn):
target
0    5174
1    1869
Name: count, dtype: int64


,CustomerID,Count,Zip Code,Latitude,Longitude,Tenure Months,Monthly Charges,Total Charges,target,Churn Score,...,Churn Reason_Lack of self-service on Website,Churn Reason_Limited range of services,Churn Reason_Long distance charges,Churn Reason_Moved,Churn Reason_Network reliability,Churn Reason_Poor expertise of online support,Churn Reason_Poor expertise of phone support,Churn Reason_Price too high,Churn Reason_Product dissatisfaction,Churn Reason_Service dissatisfaction
0,3668-QPYBK,1,90003,33.964131,-118.272783,2,53.85,108.15,1,86,...,False,False,False,False,False,False,False,False,False,False
1,9237-HQITU,1,90005,34.059281,-118.307420,2,70.70,151.65,1,67,...,False,False,False,True,False,False,False,False,False,False
2,9305-CDSKC,1,90006,34.048013,-118.293953,8,99.65,820.50,1,86,...,False,False,False,True,False,False,False,False,False,False
3,7892-POOKP,1,90010,34.062125,-118.315709,28,104.80,3046.05,1,84,...,False,False,False,True,False,False,False,False,False,False
4,0280-XJGEX,1,90015,34.039224,-118.266293,49,103.70,5036.30,1,89,...,False,False,False,False,False,False,False,False,False,False


In [10]:
# Estatísticas rápidas para conferência (dataset já pré-processado)
df.describe()

,Count,Zip Code,Latitude,Longitude,Tenure Months,Monthly Charges,Total Charges,target,Churn Score,CLTV
count,7043.0,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000
mean,1.0,93521.964646,36.282441,-119.798880,32.371149,64.761692,2279.734304,0.265370,58.699418,4400.295755
std,0.0,1865.794555,2.455723,2.157889,24.559481,30.090047,2266.794470,0.441561,21.525131,1183.057152
min,1.0,90001.000000,32.555828,-124.301372,0.000000,18.250000,0.000000,0.000000,5.000000,2003.000000
25%,1.0,92102.000000,34.030915,-121.815412,9.000000,35.500000,398.550000,0.000000,40.000000,3469.000000
50%,1.0,93552.000000,36.391777,-119.730885,29.000000,70.350000,1394.550000,0.000000,61.000000,4527.000000
75%,1.0,95351.000000,38.224869,-118.043237,55.000000,89.850000,3786.600000,1.000000,75.000000,5380.500000
max,1.0,96161.000000,41.962127,-114.192901,72.000000,118.750000,8684.800000,1.000000,100.000000,6500.000000


In [11]:
# Verificar (rapidamente) valores nulos - não deve haver após estágio 01-EDA
null_total = df.isnull().sum().sum()
print(f"Total de valores nulos no CSV pré-processado: {null_total}")
if null_total > 0:
    print(df.isnull().sum().sort_values(ascending=False).head(10))

Total de valores nulos no CSV pré-processado: 0


## 3. Feature Engineering

### Tarefa 1: Criar novas features baseadas nas existentes

In [12]:
# Criar features adicionais sobre o dataset já pré-processado (Etapa anterior 1)
df_engineered = df.copy()

# Garantir presença do alvo
y = df_engineered['target']
X = df_engineered.drop(columns=['target'])

# Criar features adicionais
eps = 1

# Dar maior peso a despesa mensal (quadrática)
df_engineered['Engineered Monthly Charges'] = df_engineered['Monthly Charges'] ** 2

# Relação entre despesa total e despesa mensal original
# Interpretação: Entender se houve muita variação entre meses anteriores e o total pago
df_engineered['charge_rel'] = df_engineered['Total Charges'] / (df_engineered['Monthly Charges'] + eps) - df_engineered['Tenure Months']
# Percentual da variação do esperado trimestral comparado com o mensal
# Interpretação: Houve muita variação nos meses recentes?


# Conferência rápida das novas features adicionadas
new_feats = ['Engineered Monthly Charges', 'charge_rel']
present = [c for c in new_feats if c in df_engineered.columns]
print(f"Novas features adicionadas ({len(present)}): {present}")

df_engineered.head(100)

Novas features adicionadas (2): ['Engineered Monthly Charges', 'charge_rel']


,CustomerID,Count,Zip Code,Latitude,Longitude,Tenure Months,Monthly Charges,Total Charges,target,Churn Score,...,Churn Reason_Long distance charges,Churn Reason_Moved,Churn Reason_Network reliability,Churn Reason_Poor expertise of online support,Churn Reason_Poor expertise of phone support,Churn Reason_Price too high,Churn Reason_Product dissatisfaction,Churn Reason_Service dissatisfaction,Engineered Monthly Charges,charge_rel
0,3668-QPYBK,1,90003,33.964131,-118.272783,2,53.85,108.15,1,86,...,False,False,False,False,False,False,False,False,2899.8225,-0.028259
1,9237-HQITU,1,90005,34.059281,-118.307420,2,70.70,151.65,1,67,...,False,True,False,False,False,False,False,False,4998.4900,0.115063
2,9305-CDSKC,1,90006,34.048013,-118.293953,8,99.65,820.50,1,86,...,False,True,False,False,False,False,False,False,9930.1225,0.152012
3,7892-POOKP,1,90010,34.062125,-118.315709,28,104.80,3046.05,1,84,...,False,True,False,False,False,False,False,False,10983.0400,0.790643
4,0280-XJGEX,1,90015,34.039224,-118.266293,49,103.70,5036.30,1,89,...,False,False,False,False,False,False,False,False,10753.6900,-0.897803
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,0259-GBZSH,1,92122,32.857230,-117.209774,2,85.65,181.50,1,73,...,False,False,False,False,False,False,True,False,7335.9225,0.094634
96,9601-BRXPO,1,92129,32.961064,-117.134917,25,104.95,2566.50,1,66,...,False,False,False,False,False,False,False,True,11014.5025,-0.776310
97,6905-NIQIN,1,92154,32.578103,-117.012975,1,50.65,50.65,1,76,...,False,False,False,False,False,False,False,False,2565.4225,-0.019361
98,5167-ZFFMM,1,92201,33.713891,-116.237257,1,90.85,90.85,1,81,...,False,False,True,False,False,False,False,False,8253.7225,-0.010887


## 4. Seleção de Features

### Tarefa 2: Selecione as features mais relevantes

In [13]:
# Dividir dados (dataset já numérico, sem OHE adicional)
y = df_engineered['target']
X = df_engineered.drop(columns=['target','CustomerID'])
feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Seleção de features usando ANOVA F-value
selector_preview = SelectKBest(f_classif, k=min(30, X_train.shape[1]))
X_train_selected = selector_preview.fit_transform(X_train, y_train)

X_test_selected = selector_preview.transform(X_test)

selected_mask = selector_preview.get_support()
selected_features = [name for name, keep in zip(feature_names, selected_mask) if keep]
print(f"\nFeatures selecionadas ({len(selected_features)}):")
for i, feat in enumerate(selected_features, 1):
    print(f"{i}. {feat}")


Features selecionadas (30):
1. Tenure Months
2. Monthly Charges
3. Total Charges
4. Churn Score
5. Dependents_Yes
6. Internet Service_Fiber optic
7. Internet Service_No
8. Online Security_No internet service
9. Online Backup_No internet service
10. Device Protection_No internet service
11. Tech Support_No internet service
12. Streaming TV_No internet service
13. Streaming Movies_No internet service
14. Contract_One year
15. Contract_Two year
16. Paperless Billing_Yes
17. Payment Method_Electronic check
18. Churn Label_Yes
19. Churn Reason_Attitude of support person
20. Churn Reason_Competitor had better devices
21. Churn Reason_Competitor made better offer
22. Churn Reason_Competitor offered higher download speeds
23. Churn Reason_Competitor offered more data
24. Churn Reason_Don't know
25. Churn Reason_Lack of self-service on Website
26. Churn Reason_Network reliability
27. Churn Reason_Price too high
28. Churn Reason_Product dissatisfaction
29. Churn Reason_Service dissatisfaction
3

## 5. Treinamento de diferentes algoritmos

### Passo a passo para o treinamento
1. Definimos um dicionário com os modelos que queremos comparar.
2. Montamos um `Pipeline` simples que inclui `StandardScaler` apenas quando o algoritmo é sensível à escala (como o SVM).
3. Treinamos cada modelo com os dados de treino e avaliamos no conjunto de teste.
4. Consolidamos as métricas em uma tabela para facilitar a comparação inicial.

In [14]:
# Comparação inicial entre diferentes algoritmos de classificação
model_configs = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Support Vector Machine": SVC(kernel="rbf", probability=True, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
}

trained_models = {}
results = []

for model_name, estimator in model_configs.items():
    steps = []
    if model_name == "Support Vector Machine":
        steps.append(("scaler", StandardScaler()))
    steps.append(("model", estimator))
    pipeline = Pipeline(steps)
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = np.nan
    if hasattr(pipeline, "predict_proba"):
        y_proba = pipeline.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_proba)
    elif hasattr(pipeline, "decision_function"):
        y_scores = pipeline.decision_function(X_test)
        roc_auc = roc_auc_score(y_test, y_scores)

    print(f"=== {model_name} ===")
    print(f"Acurácia no conjunto de teste: {accuracy:.3f}")
    print(classification_report(y_test, y_pred, target_names=["Sem Churn", "Houve Churn"]))
    print("-" * 70)

    trained_models[model_name] = pipeline
    results.append({
        "Modelo": model_name,
        "Acurácia": accuracy,
        "Precisão": precision,
        "Recall": recall,
        "F1": f1,
        "ROC AUC": roc_auc,
    })

results_df = pd.DataFrame(results)
results_df.sort_values(by="Acurácia", ascending=False).reset_index(drop=True).round(3)

=== Decision Tree ===
Acurácia no conjunto de teste: 1.000
              precision    recall  f1-score   support

   Sem Churn       1.00      1.00      1.00      1035
 Houve Churn       1.00      1.00      1.00       374

    accuracy                           1.00      1409
   macro avg       1.00      1.00      1.00      1409
weighted avg       1.00      1.00      1.00      1409

----------------------------------------------------------------------
=== Random Forest ===
Acurácia no conjunto de teste: 1.000
              precision    recall  f1-score   support

   Sem Churn       1.00      1.00      1.00      1035
 Houve Churn       1.00      1.00      1.00       374

    accuracy                           1.00      1409
   macro avg       1.00      1.00      1.00      1409
weighted avg       1.00      1.00      1.00      1409

----------------------------------------------------------------------
=== Support Vector Machine ===
Acurácia no conjunto de teste: 1.000
              prec

,Modelo,Acurácia,Precisão,Recall,F1,ROC AUC
0,Decision Tree,1.0,1.0,1.0,1.0,1.0
1,Random Forest,1.0,1.0,1.0,1.0,1.0
2,Support Vector Machine,1.0,1.0,1.0,1.0,1.0
3,Gradient Boosting,1.0,1.0,1.0,1.0,1.0
